# deepMaze — train DRQN + DTQN on Colab (Drive-backed)

Mounts Google Drive, clones the repo into `/content/deepMaze`, trains **DRQN** then **DTQN** on a 30×60 multi-treasure + lava maze, and persists everything (MLflow runs, model weights, bundles, replays) to Drive at `${DRIVE_BASE}`.

**No GCP required** — MLflow uses a `file://` tracking store on Drive.

After the run finishes:
- `${DRIVE_BASE}/mlruns/` — full MLflow experiment store (open with `mlflow ui --backend-store-uri ${DRIVE_BASE}/mlruns`)
- `${DRIVE_BASE}/assets/<run_name>/` — ready-to-inference bundles (`config.json` + `model.pt` + `viz/replay.webp`). Copy to your local `assets/` to use with `python web/server.py`.

> **Heads up:** 30×60 with `partial_view=3` is hard — the agent has a 7×7 window in a ~1800-cell maze. Training takes ~1–2 h on T4 for DRQN at 6 k episodes. Drop `MAZE_WIDTH/HEIGHT` for faster smoke-tests.


## 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


## 2 — Config (Colab form)

In [ ]:
# @title Config { run: "auto" }
# ╭─ Repo + paths ──────────────────────────────────────────────────────────╮
REPO_URL    = "https://github.com/juan-garassino/deepMaze.git"  # @param {type:"string"}
REPO_BRANCH = "main"                                             # @param {type:"string"}
DRIVE_BASE  = "/content/drive/MyDrive/deepMaze"                  # @param {type:"string"}

# ╭─ Run ──────────────────────────────────────────────────────────────────╮
AGENTS_TO_RUN     = "drqn,dtqn"  # @param {type:"string"}
RUN_TAG           = "v1"          # @param {type:"string"}
MLFLOW_EXPERIMENT = "deepmaze"   # @param {type:"string"}
SEED              = 0             # @param {type:"integer"}

# ╭─ Maze ─────────────────────────────────────────────────────────────────╮
# WHY these defaults: partial_view=5 gives an 11x11 window; COLLECT_ALL=False
# ends episode on first treasure → fast positive signal for the buffer;
# more treasures + denser walls = richer learning landscape.
MAZE_WIDTH   = 30    # @param {type:"integer"}
MAZE_HEIGHT  = 60    # @param {type:"integer"}
GENERATOR    = "dfs" # @param ["open", "dfs", "random"]
DENSITY      = 0.2   # @param {type:"number"}
N_TREASURES  = 10    # @param {type:"integer"}
N_LAVA       = 2     # @param {type:"integer"}
COLLECT_ALL  = False # @param {type:"boolean"}
PARTIAL      = 5     # @param {type:"integer"}

# ╭─ Generalization ───────────────────────────────────────────────────────╮
# REGENERATE_EVERY = N → re-roll the maze layout every N episodes during
# training so the agent learns "how to solve mazes" instead of memorizing
# one. 1 = brand-new maze each episode (strongest generalization). 0 = off.
# EVAL_REGENERATE = True → eval episodes also use fresh mazes (honest score).
REGENERATE_EVERY = 1     # @param {type:"integer"}
EVAL_REGENERATE  = True  # @param {type:"boolean"}

# ╭─ Training budget ──────────────────────────────────────────────────────╮
NUM_EPISODES  = 3000  # @param {type:"integer"}
MAX_STEPS     = 600   # @param {type:"integer"}
EVAL_EPISODES = 50    # @param {type:"integer"}

# ╭─ Agent hyperparameter overrides (0 / 0.0 = use repo default) ──────────╮
# exploration_decay slowed 10x: 0.999995 over 600-step episodes gives
# per-ep factor ~0.997, so ε reaches 0.05 around episode 1000.
EXPLORATION_DECAY = 0.999995  # @param {type:"number"}
BUFFER_CAPACITY   = 50000     # @param {type:"integer"}
LEARNING_RATE     = 0.0       # @param {type:"number"}
BATCH_SIZE        = 0         # @param {type:"integer"}
SEQ_LEN           = 0         # @param {type:"integer"}
TARGET_SYNC       = 0         # @param {type:"integer"}
LSTM_HIDDEN       = 0         # @param {type:"integer"}

# ╭─ Display + showcase ───────────────────────────────────────────────────╮
PRINT_EVERY     = 25   # @param {type:"integer"}
SHOWCASE_EVERY  = 250  # @param {type:"integer"}
WINDOW          = 100  # @param {type:"integer"}
SHOWCASE_SPRITE = 12   # @param {type:"integer"}
SHOWCASE_FRAMES = 300  # @param {type:"integer"}


## 3 — Clone repo + install deps

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

WORK = pathlib.Path("/content/deepMaze")
# Always step out of WORK before nuking it — otherwise the shell's cwd
# ends up pointing at a deleted inode and every subsequent subprocess
# call dies with "cannot access parent directories".
os.chdir("/content")
if WORK.exists():
    shutil.rmtree(WORK)
subprocess.check_call(["git", "clone", "--depth=1", "-b", REPO_BRANCH, REPO_URL, str(WORK)])

os.chdir(WORK)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", "requirements.txt", "mlflow==2.18.*"])

for sub in ("agents", "environment", "training", "utils", "config"):
    p = str(WORK / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

print("repo:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip(),
      "branch:", REPO_BRANCH)


## 4 — Drive paths + MLflow tracking URI

In [ ]:
from pathlib import Path

DRIVE_BASE_P = Path(DRIVE_BASE)
MLRUNS_DIR   = DRIVE_BASE_P / "mlruns"
ASSETS_DIR   = DRIVE_BASE_P / "assets"

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = f"file://{MLRUNS_DIR.as_posix()}"

import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print("tracking:", MLFLOW_TRACKING_URI)
print("assets  :", ASSETS_DIR)


## 5 — Train one agent (function)

In [ ]:
"""Generic training driver. Reads ALL knobs from cell-4 globals — do not
edit this cell during normal iteration. Tweak cell 4 instead."""

import json
import shutil
import time
from collections import deque
from pathlib import Path

import mlflow
import torch

from maze import MazeEnvironment, RenderMaze
from recorders import ReplayRecorder
from seeding import seed_everything
from train import create_agent, evaluate_agent, simulate_episode, train_agent
from viz_events import EpisodeEvent, EventBus

try:
    from IPython.display import Image as _IPImage, display as _ipy_display
except Exception:
    _IPImage = None
    def _ipy_display(*a, **k): pass


def _fmt_eta(s: float) -> str:
    if s < 90:
        return f"{s:.0f}s"
    if s < 3600:
        return f"{s/60:.1f}m"
    return f"{s/3600:.2f}h"


def _hr(c: str = "─", w: int = 78) -> str:
    return c * w


def _agent_overrides(agent_type: str) -> dict:
    """Filter zeros — repo defaults win when the form is left blank."""
    candidates = {
        "exploration_decay": EXPLORATION_DECAY,
        "learning_rate":     LEARNING_RATE,
        "batch_size":        BATCH_SIZE,
        "seq_len":           SEQ_LEN,
        "target_sync":       TARGET_SYNC,
    }
    if agent_type in ("drqn", "dtqn", "dqn"):
        candidates["buffer_capacity"] = BUFFER_CAPACITY
    if agent_type == "drqn":
        candidates["lstm_hidden"] = LSTM_HIDDEN
    return {k: v for k, v in candidates.items() if v}


def train_one(agent_type: str, run_name: str) -> dict:
    title = f"  {agent_type.upper()}  —  {run_name}  "
    print(_hr("━"))
    print(" " * max(0, (78 - len(title)) // 2) + title)
    print(_hr("━"))

    seed_everything(SEED)

    env = MazeEnvironment(
        width=MAZE_WIDTH, height=MAZE_HEIGHT, density=DENSITY,
        generator=GENERATOR, n_lava=N_LAVA, n_treasures=N_TREASURES,
        collect_all=COLLECT_ALL, partial_view=PARTIAL, seed=SEED,
    )
    overrides = _agent_overrides(agent_type)
    agent = create_agent(agent_type, env, **overrides)
    print(f"  agent       : {type(agent).__name__}")
    print(f"  maze        : {MAZE_WIDTH}x{MAZE_HEIGHT}  generator={GENERATOR}  density={DENSITY}")
    print(f"  treasures   : {N_TREASURES}  lava: {N_LAVA}  collect_all: {COLLECT_ALL}  partial_view: {PARTIAL}")
    print(f"  budget      : {NUM_EPISODES} episodes  max_steps={MAX_STEPS}")
    print(f"  regen every : {REGENERATE_EVERY or 'off (single fixed maze)'}   eval_regen: {EVAL_REGENERATE}")
    print(f"  overrides   : {overrides or '(none — using repo defaults)'}")
    print(_hr())

    showcase_dir   = Path(f"/content/showcase/{run_name}")
    drive_showcase = ASSETS_DIR.parent / "showcase" / run_name
    showcase_dir.mkdir(parents=True, exist_ok=True)
    drive_showcase.mkdir(parents=True, exist_ok=True)
    sprites = RenderMaze.placeholder_sprites(SHOWCASE_SPRITE)

    def render_snapshot(ep: int) -> Path:
        agent.set_deterministic(True)
        try:
            _, positions, _, _ = simulate_episode(env, agent, max_steps=MAX_STEPS, at_start=True)
        finally:
            agent.set_deterministic(False)
        full_frames = [env.maze.copy() for _ in positions]
        rm = RenderMaze(sprites)
        ReplayRecorder(rm).feed(full_frames, positions, None)
        out = showcase_dir / f"ep{ep:05d}.webp"
        rm.save(str(out), fmt="webp", sprite_size=SHOWCASE_SPRITE, max_frames=SHOWCASE_FRAMES)
        try:
            shutil.copy(out, drive_showcase / out.name)
        except Exception as e:
            print(f"  [showcase] drive copy failed: {e}")
        return out

    bus = EventBus()

    def _on_ep_mlflow(ev: EpisodeEvent):
        mlflow.log_metrics(
            {"episode_reward": ev.total_reward,
             "episode_length": ev.length,
             "epsilon": ev.epsilon},
            step=ev.episode,
        )

    reward_buf  = deque(maxlen=WINDOW)
    length_buf  = deque(maxlen=WINDOW)
    success_buf = deque(maxlen=WINDOW)
    t0 = time.time()

    def _on_ep_print(ev: EpisodeEvent):
        reward_buf.append(ev.total_reward)
        length_buf.append(ev.length)
        success_buf.append(1 if ev.total_reward > 0 else 0)

        elapsed   = time.time() - t0
        eps_per_s = (ev.episode + 1) / max(elapsed, 1e-6)
        eta       = (NUM_EPISODES - ev.episode - 1) / max(eps_per_s, 1e-6)

        if ev.episode % PRINT_EVERY == 0 or ev.episode == NUM_EPISODES:
            avg_r = sum(reward_buf) / len(reward_buf)
            succ  = 100.0 * sum(success_buf) / len(success_buf)
            extra = f" loss={ev.loss:.4f}" if ev.loss is not None else ""
            print(
                f"  ep {ev.episode:>5}/{NUM_EPISODES}  "
                f"R={ev.total_reward:+6.2f}  R̄{WINDOW}={avg_r:+6.2f}  succ%={succ:5.1f}  "
                f"len={ev.length:>4}  ε={ev.epsilon:.3f}{extra}  "
                f"[{eps_per_s:5.1f} ep/s  ETA {_fmt_eta(eta)}]",
                flush=True,
            )

        if ev.episode > 0 and (ev.episode % SHOWCASE_EVERY == 0):
            avg_r = sum(reward_buf) / len(reward_buf)
            avg_l = sum(length_buf) / len(length_buf)
            succ  = 100.0 * sum(success_buf) / len(success_buf)
            print()
            print(_hr("╌"))
            print(f"  ▣  SHOWCASE  ep {ev.episode}/{NUM_EPISODES}  ({100*ev.episode/NUM_EPISODES:.1f}%)")
            print(f"     last {WINDOW}: R̄={avg_r:+.2f}   len̄={avg_l:.0f}   success={succ:.1f}%")
            print(f"     elapsed {_fmt_eta(elapsed)}   ETA {_fmt_eta(eta)}   throughput {eps_per_s:.1f} ep/s")
            snap = render_snapshot(ev.episode)
            print(f"     replay → {snap.name}  ({snap.stat().st_size//1024} KB)")
            if _IPImage is not None:
                _ipy_display(_IPImage(filename=str(snap)))
            print(_hr("╌"))
            print()

    bus.subscribe(_on_ep_mlflow)
    bus.subscribe(_on_ep_print)

    params = dict(
        agent_type=agent_type, run_name=run_name,
        maze_width=MAZE_WIDTH, maze_height=MAZE_HEIGHT,
        generator=GENERATOR, density=DENSITY,
        n_treasures=N_TREASURES, n_lava=N_LAVA, collect_all=COLLECT_ALL,
        partial=PARTIAL, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
        eval_episodes=EVAL_EPISODES, seed=SEED,
        regenerate_every=REGENERATE_EVERY, eval_regenerate=EVAL_REGENERATE,
        **overrides,
    )

    train_t0 = time.time()
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)
        train_agent(env, agent, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS, bus=bus,
                    regenerate_every=(REGENERATE_EVERY or None))
        mean_reward, mean_length, success_rate = evaluate_agent(
            env, agent, num_episodes=EVAL_EPISODES, max_steps=MAX_STEPS,
            regenerate_each=EVAL_REGENERATE,
        )
        mlflow.log_metrics({
            "eval_mean_reward": mean_reward,
            "eval_mean_length": mean_length,
            "eval_success_rate": success_rate,
        })
        run_id = run.info.run_id

    print()
    print(_hr("━"))
    print(f"  ✓ {agent_type.upper()} done in {_fmt_eta(time.time() - train_t0)}")
    print(f"    eval: reward {mean_reward:+.2f}   length {mean_length:.0f}   success {success_rate:.1%}")
    print(_hr("━"))

    local_out = Path(f"assets/{run_name}")
    drive_out = ASSETS_DIR / run_name
    for out in (local_out, drive_out):
        (out / "viz").mkdir(parents=True, exist_ok=True)

    config = dict(
        agent_type=agent_type, net=None,
        maze_width=MAZE_WIDTH, maze_height=MAZE_HEIGHT,
        n_agents=1, density=DENSITY, generator=GENERATOR,
        no_ensure_solvable=False, n_lava=N_LAVA, lava_reward=-1.0,
        partial=PARTIAL, n_treasures=N_TREASURES, collect_all=COLLECT_ALL,
        seed=SEED, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
        eval_episodes=EVAL_EPISODES, learning_rate=None, discount_factor=0.99,
        image_path=None, sprite_files=["sprites.png"], sprite_size=32,
        replay_fmt="webp", frame_skip=1, max_frames=None,
        policy_snapshot_every=50, live=False, live_web=False, web_port=8000,
        run_id=run_id, run_name=run_name,
        random_start=False, resume=None, eval_maze="same", eval_seeds=1,
    )
    (local_out / "config.json").write_text(json.dumps(config, indent=2))

    module = getattr(agent, "model", None) or getattr(agent, "ac", None)
    torch.save(module.state_dict(), local_out / "model.pt")

    final_snap = render_snapshot(NUM_EPISODES)
    shutil.copy(final_snap, local_out / "viz" / "replay.webp")
    print(f"  final replay → {local_out / 'viz' / 'replay.webp'}")
    if _IPImage is not None:
        _ipy_display(_IPImage(filename=str(local_out / "viz" / "replay.webp")))

    if drive_out.exists():
        shutil.rmtree(drive_out)
    shutil.copytree(local_out, drive_out)

    with mlflow.start_run(run_id=run_id):
        mlflow.log_artifacts(str(local_out), artifact_path=f"assets/{run_name}")

    print(f"  bundle → {drive_out}")
    return {
        "agent_type": agent_type,
        "run_name": run_name,
        "run_id": run_id,
        "eval_success_rate": success_rate,
        "eval_mean_reward": mean_reward,
        "drive_path": str(drive_out),
        "showcase_dir": str(drive_showcase),
    }


## 6 — Train DRQN then DTQN sequentially

In [ ]:
agents = [a.strip() for a in AGENTS_TO_RUN.split(",") if a.strip()]
results = []
for agent_type in agents:
    run_name = f"{agent_type}_{RUN_TAG}"
    results.append(train_one(agent_type, run_name))

print("\n=== summary ===")
for r in results:
    print(f"  {r['agent_type']:5s}  {r['run_name']:20s}  success={r['eval_success_rate']:.2%}  → {r['drive_path']}")


## 7 — Use the bundles locally

On your laptop:

```bash
# Copy a bundle from Drive into the repo's assets/
rsync -a "<google-drive-folder>/deepMaze/assets/drqn_v1/" assets/drqn_v1/

# Then either:
python web/server.py --port 8000    # → http://localhost:8000  → pretrained dropdown
# or:
python main.py --agent_type drqn --resume assets/drqn_v1/model.pt --num_episodes 0
```

To browse MLflow runs locally after copying `mlruns/` down from Drive:

```bash
mlflow ui --backend-store-uri "file://$(pwd)/mlruns"
```
